In [116]:
from label_studio_sdk.label_interface import LabelInterface
from label_studio_sdk.label_interface.create import labels
from label_studio_sdk.actions import ActionsCreateRequestSelectedItemsExcluded
from ls_helper import *
import requests

In [ ]:
# Define the URL where Label Studio is accessible and the API key for your user account
LABEL_STUDIO_URL = 'http://localhost:8080'
LABEL_STUDIO_ML_BACKEND_URL = 'http://localhost:9090'

# API key is available at the Account & Settings > Access Tokens page in Label Studio UI
API_KEY = '<your api key>'

# Import the SDK and the client module
from label_studio_sdk.client import LabelStudio

# Connect to the Label Studio API and check the connection
ls = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

In [ ]:
FLAIR_TEST_PROJECT = 'flair-ner-test'
RECREATE_PROJECT = False
LOAD_TASKS_FROM_LOCAL_FILE = True
LOCAL_TASKS_FILE='sample_data/flair_ner.txt'

# Define labeling interface
label_config = LabelInterface.create({
    'text': 'Text',
    'label': labels(['PER', 'ORG', 'LOC', 'MISC'])
})

In [119]:
# Create a project with the specified title and labeling configuration
project = get_or_create_project(ls, FLAIR_TEST_PROJECT, label_config, RECREATE_PROJECT)
print(project)

id=58 title='flair-ner-test' description='' label_config='<View>\n  <Text name="text" value="$text"/>\n  <Labels name="label" toName="text">\n    <Label value="PER"/>\n    <Label value="ORG"/>\n    <Label value="LOC"/>\n    <Label value="MISC"/>\n  </Labels>\n</View>' expert_instruction='' show_instruction=False show_skip_button=True enable_empty_annotation=True show_annotation_history=False reveal_preannotations_interactively=False show_collab_predictions=True maximum_annotations=1 color='#FFFFFF' control_weights={'label': {'overall': 1.0, 'type': 'Labels', 'labels': {'PER': 1.0, 'ORG': 1.0, 'LOC': 1.0, 'MISC': 1.0}}} organization=1 is_published=False model_version='' is_draft=False created_by={'id': 2, 'first_name': '', 'last_name': '', 'email': 'abc@gmail.com', 'avatar': None} created_at='2025-03-20T21:35:42.549855Z' min_annotations_to_start_training=0 start_training_on_annotation_update=False num_tasks_with_annotations=None task_number=None useful_annotation_number=None ground_trut

In [120]:
if RECREATE_PROJECT:
    if LOAD_TASKS_FROM_LOCAL_FILE:
        with open(LOCAL_TASKS_FILE, 'r') as file:
            for line in file:
                ls.projects.import_tasks(
                    id=project.id,
                    request=[
                        {"text": line.strip()}
                    ]
                )
    else:
        # Create some sample tasks
        ls.projects.import_tasks(
            id=project.id,
            request=[
                {"text": "Copenhagen Denmark"},
                {"text": "Washington United States"},
                {"text": "Paris France"},
            ]
        )

In [121]:
# Create and connect the ML Backend with the project
flair_model = get_or_create_model(
    ls,
    title='FLAIR NER',
    description='A model to perform NER using Flair',
    url=LABEL_STUDIO_ML_BACKEND_URL,
    project_id=project.id,
    is_interactive=True
)

In [122]:
# Create annotations using ML backend for all the imported tasks 
ls.actions.create(
            id="retrieve_tasks_predictions",
            project=project.id,
            selected_items=ActionsCreateRequestSelectedItemsExcluded(
                all_=True
            )
        )

In [123]:
# Verify the annotations predicted by the ML backend and examine confidence score
for task in ls.tasks.list(project=project.id):
    print(task)

id=6 predictions=[{'id': 119, 'result': [{'from_name': 'label', 'score': 0.7555165588855743, 'to_name': 'text', 'type': 'labels', 'value': {'end': 11, 'labels': ['LOC'], 'start': 0, 'text': 'Tokyo Japan'}}], 'model_version': 'Flair-v0.0.1', 'created_ago': '0\xa0minutes', 'score': 0.7555165588855743, 'cluster': None, 'neighbors': None, 'mislabeling': 0.0, 'created_at': '2025-03-20T21:36:10.585071Z', 'updated_at': '2025-03-20T21:36:10.585091Z', 'model': None, 'model_run': None, 'task': 6, 'project': 58}] annotations=[] drafts=[] annotators=[] inner_id=1 cancelled_annotations=0 total_annotations=0 total_predictions=1 completed_at=None file_upload=None storage_filename=None avg_lead_time=None draft_exists=False updated_by=[{'user_id': 2}] data={'text': 'Tokyo Japan'} meta={} created_at=datetime.datetime(2025, 3, 20, 21, 35, 42, 570512, tzinfo=TzInfo(UTC)) updated_at=datetime.datetime(2025, 3, 20, 21, 36, 10, 579390, tzinfo=TzInfo(UTC)) is_labeled=False overlap=1.0 comment_count=0 unresolve

In [ ]:
# Get the export of the annotations
url = LABEL_STUDIO_URL + "/api/projects/" + str(project.id) + "/export"
print(url)
token = 'Token ' + API_KEY
print(token)
Headers = { 'Authorization' : token }
response = requests.get(url, headers=Headers )
print(response.json())

http://localhost:8080/api/projects/58/export
Token 3febbb103045e50e96dae2499df316299d73220e
[{'id': 6, 'annotations': [{'id': 168, 'completed_by': 2, 'result': [{'value': {'start': 0, 'end': 11, 'text': 'Tokyo Japan', 'labels': ['LOC']}, 'id': 'xjan9CYW1Z', 'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'origin': 'prediction', 'score': 0.7555165588855743}, {'value': {'start': 0, 'end': 11, 'text': 'Tokyo Japan', 'labels': ['MISC']}, 'id': 'N-E7I73uX5', 'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'origin': 'manual'}, {'value': {'start': 0, 'end': 11, 'text': 'Tokyo Japan', 'labels': ['MISC']}, 'id': '6Rp5tftnSo', 'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'origin': 'manual', 'score': 0.7555165588855743}], 'was_cancelled': False, 'ground_truth': False, 'created_at': '2025-03-20T21:37:50.191753Z', 'updated_at': '2025-03-20T21:37:50.191771Z', 'draft_created_at': '2025-03-20T21:37:47.240870Z', 'lead_time': 17.33, 'prediction': {}, 'result_count': 3